# 🏠 House Price Prediction — Backend Notebook
**This notebook trains the ML model and saves it as `model.pkl`**  
The saved model is then loaded and served by `app.py` (Flask Backend)  
The frontend `frontend.py` (Streamlit) calls the Flask API to get predictions

---
### 🗂️ Project File Structure
```
project/
├── backend_notebook.ipynb  ← This file (train & save model)
├── app.py                  ← Flask Backend (API server)
├── frontend.py             ← Streamlit Frontend (UI)
├── model.pkl               ← Saved trained model
└── Housing_Cleaned.csv     ← Dataset
```
---

## 📦 Step 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print('✅ All libraries imported successfully!')

## 📂 Step 2 — Load & Preview Dataset

In [ ]:
df = pd.read_csv('Housing_Cleaned.csv')
print('Shape:', df.shape)
df.head()

## 🔀 Step 3 — Define Features & Target, Split Data

In [ ]:
# Features (X) and Target (y)
FEATURES = [
    'Area_sqft', 'Bedrooms', 'Bathrooms', 'Stories',
    'MainRoad_Access', 'GuestRoom', 'Basement',
    'HotWater_Heating', 'AirConditioning',
    'Parking_Spaces', 'Preferred_Area', 'Furnishing_Status'
]

X = df[FEATURES]
y = df['Price']

# 80% Train, 20% Test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training samples : {X_train.shape[0]}')
print(f'Testing  samples : {X_test.shape[0]}')
print(f'Features used    : {FEATURES}')

## 🤖 Step 4 — Train Linear Regression Model

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
mae    = mean_absolute_error(y_test, y_pred)
rmse   = np.sqrt(mean_squared_error(y_test, y_pred))
r2     = r2_score(y_test, y_pred)

print('=' * 40)
print('   MODEL EVALUATION')
print('=' * 40)
print(f'  MAE  : ₹{mae:,.0f}')
print(f'  RMSE : ₹{rmse:,.0f}')
print(f'  R²   : {r2:.4f}')
print('=' * 40)

## 💾 Step 5 — Save Model as `model.pkl`
> This `.pkl` file is what the Flask backend (`app.py`) loads to make predictions.

In [ ]:
with open('model.pkl', 'wb') as f:
    pickle.dump(model, f)

print('✅ model.pkl saved successfully!')
print('   The Flask backend will now load this file to serve predictions.')

## ✅ Step 6 — Test the Saved Model (verify it works)

In [ ]:
# Load the saved model back
with open('model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)

# Sample house input (same format Flask will receive)
sample = pd.DataFrame([{
    'Area_sqft'        : 3000,
    'Bedrooms'         : 3,
    'Bathrooms'        : 2,
    'Stories'          : 2,
    'MainRoad_Access'  : 1,
    'GuestRoom'        : 0,
    'Basement'         : 1,
    'HotWater_Heating' : 0,
    'AirConditioning'  : 1,
    'Parking_Spaces'   : 1,
    'Preferred_Area'   : 1,
    'Furnishing_Status': 1
}])

prediction = loaded_model.predict(sample)[0]
print(f'₹ Predicted Price : ₹{prediction:,.0f}')
print(f'₹ In Lakhs        : ₹{prediction/1e5:.2f} Lakhs')
print()
print('✅ Model is working correctly and ready for Flask backend!')

## 🚀 Step 7 — How to Run the Full Project

### 1️⃣ Install requirements (run once in terminal)
```bash
pip install flask streamlit requests scikit-learn pandas numpy
```

### 2️⃣ Start the Flask Backend
```bash
python app.py
```
> Server starts at: `http://localhost:5000`

### 3️⃣ Start the Streamlit Frontend (new terminal)
```bash
streamlit run frontend.py
```
> Opens in browser at: `http://localhost:8501`

### 4️⃣ Use the App
- Fill in house details in the browser UI
- Click **Predict Price**
- Frontend sends data → Flask backend → Model predicts → Result shown in UI

---
```
User (Browser UI)
      ↓ fills form
frontend.py (Streamlit)
      ↓ POST /predict
app.py (Flask API)
      ↓ loads
model.pkl (Trained Model)
      ↓ returns price
frontend.py → shows result to user
```